# REGENIE_PREP

This has 3 main functions:

1) Generate 5 common principal components via FlashPCA2 after the sample + variant filtering from array data

2) Define HCM cases in the 303,927 samples passing sample-level QC
3) Define non-HCM controls in the 303,927 samples passing sample-level QC

In [ ]:
%%capture
## Python Package Import
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

##Ensuring dsub is up to date
!pip3 install --upgrade dsub

#Defining environment variables
# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_ID={LINE_COUNT_JOB_ID}
## Defining necessary pathways
my_bucket = os.environ['WORKSPACE_BUCKET']
project_name='HCM_GWAS_V2'
## Setting for running dsub jobs
pd.set_option('display.max_colwidth', 0)

USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env USER_NAME={USER_NAME}
%env PROJECT_NAME=HCM_GWAS_V2

In [ ]:
%%capture
JOB_NAME='REGENIE_PREP'

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_NAME={JOB_NAME}
%env PHENO_FILEPATH={PHENO_FILEPATH}
%env COV_FILEPATH={COV_FILEPATH}

## Analysis Results Folder 
line_count_results_folder = os.path.join(
    os.getenv('WORKSPACE_BUCKET'),
    'dsub',
    'results',
    os.getenv('PROJECT_NAME'),
    JOB_NAME,
    datetime.now().strftime('%Y%m%d'))

## Where the output files will go
output_files = os.path.join(line_count_results_folder)

OUTPUT_FILES = output_files

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env OUTPUT_FILES={OUTPUT_FILES}

## FlashPCA2

This runs FlashPCA2 to generate PCs from the sample/variant-level data with a first step of LD pruning the variants passing the variant-level array QC.

1) Prepare the plink bfiles after filtering for sample/variants passing QC -> LD pruning
2) Perform FlashPCA2
3) Plot PC plots with colouring by self-reported race.

In [ ]:
filename='aux_scripts/3_REGENIE_PREP_flashpca2_prep.sh'

script = '''
set -o pipefail 
set -o errexit

plink2 \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids}" \
    --extract "${variant_qc_pass_vars}" \
    --exclude bed1 "${exclusion_region}" \
    --indep-pairwise 1000 50 0.05 \
    --memory 30000 \
    --out array_flashpca2_varprep
    
plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids}" \
    --extract array_flashpca2_varprep.prune.in \
    --make-bed \
    --memory 30000\
    --out array_flashpca2_ldpruned

export output_prune_in="array_flashpca2_varprep.prune.in"
export output_prune_out="array_flashpca2_varprep.prune.out"
export log=array_flashpca2_varprep.log

export bed=array_flashpca2_ldpruned.bed
export bim=array_flashpca2_ldpruned.bim
export fam=array_flashpca2_ldpruned.fam
export log2=array_flashpca2_ldpruned.log

mv ${output_prune_in} ${output_prune_out} ${log} ${bed} ${bim} ${fam} ${log2} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
#Upload to GCP Bucket
!gsutil cp aux_scripts/3_REGENIE_PREP_flashpca2_prep.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" 
SCRIPT_NAME="3_REGENIE_PREP_flashpca2_prep.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"


dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "flashPCA2_prep" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input sample_qc_pass_ids="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv" \
    --input variant_qc_pass_vars="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/2_VARIANT_QC/array/array_variantqc_pass_final_vars.txt" \
    --input exclusion_region="${WORKSPACE_BUCKET}/ACAF_VARIANT_QC/1_input/FlashPCA2/exclusion_regions_hg38.txt" \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251104/*.log ../3_output/3_REGENIE_PREP/1_flashPCA2/
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251104/array_flashpca2_ldpruned.bim ../3_output/3_REGENIE_PREP/1_flashPCA2/

In [ ]:
!wc -l ../3_output/3_REGENIE_PREP/1_flashPCA2/array_flashpca2_ldpruned.bim 

In [ ]:
filename='aux_scripts/3_REGENIE_PREP_flashPCA2.sh'

script = '''
set -o pipefail 
set -o errexit

#This defines the actual bed_prefix, assuming localisation of the input bed/bim/fam files at the /mnt/data/input/ folder using the same directory structure
bfile_prefix="/mnt/data/input/gs/${MY_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251104/array_flashpca2_ldpruned"

~/flashpca/flashpca --bfile $bfile_prefix 

export eigenvector=eigenvectors.txt
export pcs=pcs.txt
export eigenvalues=eigenvalues.txt
export pve=pve.txt

mv ${eigenvector} ${pcs} ${eigenvalues} ${pve} -t $OUTPUT_PATH
'''

with open(filename,'w') as fp:
    fp.write(script)
    
#Upload to GCP Bucket
!gsutil cp aux_scripts/3_REGENIE_PREP_flashPCA2.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

# For AoU RWB projects network name is "network".
AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-16" #Needs 32GB of RAM for the biggest chromosomes in REGENIE
SCRIPT_NAME="3_REGENIE_PREP_flashPCA2.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"
BUCKET_BASENAME="${WORKSPACE_BUCKET#*gs://}" #Removes the gs:// prefix

dsub \
--provider google-batch \
--user-project "${GOOGLE_PROJECT}" \
--project "${GOOGLE_PROJECT}" \
--image "${ARTIFACT_REGISTRY_DOCKER_REPO}/plenzini/flashpca:latest" \
--network "${AOU_NETWORK}" \
--subnetwork "${AOU_SUBNETWORK}" \
--service-account "$(gcloud config get-value account)" \
--user "${DSUB_USER_NAME}" \
--regions us-central1 \
--logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
"$@" \
--disk-size 1000 \
--boot-disk-size 100 \
--machine-type ${MACHINE_TYPE} \
--name "flashpca2_array" \
--script "${BASH_SCRIPT}" \
--env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
--env MY_BUCKET=${BUCKET_BASENAME} \
--input bedfile="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251104/array_flashpca2_ldpruned.bed" \
--input bimfile="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251104/array_flashpca2_ldpruned.bim" \
--input famfile="${WORKSPACE_BUCKET}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251104/array_flashpca2_ldpruned.fam" \
--use-private-address \
--output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251104/*.txt ../3_output/3_REGENIE_PREP/1_flashPCA2/

In [ ]:
#Plot the PCs against the self-reported ancestry of the individuals

## Import in the data
data = pd.read_csv('../1_input/0_cohort_tsvs/HCM_GWAS_START.tsv', sep='\t')
sampleqc_passed_ids = pd.read_csv('../3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv', sep='\t')

data = data[data['person_id'].isin(sampleqc_passed_ids['IID'])]
print(data.shape)

#Plot with the self-reported ethnicity 
pcs = pd.read_csv('../3_output/3_REGENIE_PREP/1_flashPCA2/pcs.txt', sep='\t')
pcs = pcs.drop(columns=['FID']).rename(columns={'IID':'person_id'})
data = data.merge(pcs, how='left', on='person_id')

data['race'] = data['race'].replace(
    ['None Indicated', 'PMI: Skip', 'I prefer not to answer'],
    np.nan
)

data['race'] = data['race'].replace(
    ['More than one population'],
    'Admixed'
)

data['race'] = data['race'].replace(
    ['None of these'],
    'Other'
)

data.to_csv('../3_output/3_REGENIE_PREP/1_flashPCA2/hcm_gwas_sample_qc_pass_PCs.tsv',sep='\t',index=False)
data['race'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plot_pc_biplots(df, output_folder):
    """
    Generates and displays PC biplots for specified combinations.

    This function creates a 3x1 grid of scatter plots for:
    1. PC1 vs. PC2
    2. PC1 vs. PC3
    3. PC2 vs. PC3

    Points in each plot are colored based on the 'race' column.

    Args:
        df (pd.DataFrame): A DataFrame containing at least 'race', 'PC1', 
                           'PC2', and 'PC3' columns.
    """
    # Set the aesthetic style of the plots
    sns.set_style("whitegrid")

    # Create a figure and a set of subplots
    # 1 row, 3 columns for the three plots
    fig, axes = plt.subplots(1, 3, figsize=(27, 9))
    fig.suptitle('Principal Component Analysis Biplots', fontsize=16)

    # List of PC combinations to plot
    pc_combinations = [('PC1', 'PC2'), ('PC1', 'PC3'), ('PC2', 'PC3')]

    for i, (pc_x, pc_y) in enumerate(pc_combinations):
        ax = axes[i]
        sns.scatterplot(data=df, x=pc_x, y=pc_y, hue='race', ax=ax, s=5, alpha=0.1)
        ax.set_title(f'{pc_x} vs. {pc_y}', fontsize=12)
#         plt.figtext(0.5, 0.01, f"{df.dropna(axis=0,subset='race').shape[0]} non-NA ethnicity individuals", wrap=True, horizontalalignment='center', fontsize=12)
        ax.set_xlabel(f'Principal Component {pc_x[-1]}')
        ax.set_ylabel(f'Principal Component {pc_y[-1]}')
        ax.legend(title='Race')

    # Adjust layout to prevent overlap and display the plot
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f'{output_folder}pcs_vs_selfreportedeth.png', dpi=600)
    plt.show()


plot_pc_biplots(data, '../3_output/3_REGENIE_PREP/1_flashPCA2/plots/')

## Step 1 Variant Preparation

This creates subsetted versions of the hard-called SNPs via LD pruning for Step 1 of REGENIE.

In [ ]:
filename='aux_scripts/3_REGENIE_PREP_step1_varprep.sh'

script = '''
set -o pipefail 
set -o errexit

plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids}" \
    --extract "${variant_qc_pass_vars}" \
    --indep-pairwise 1000 50 0.5 \
    --memory 30000 \
    --out array_regenie_step1_varprep
    
plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids}" \
    --extract array_regenie_step1_varprep.prune.in \
    --make-bed \
    --memory 30000 \
    --out array_regenie_step1_ldpruned_varprep

export output_prune_in="array_regenie_step1_varprep.prune.in"
export output_prune_out="array_regenie_step1_varprep.prune.out"
export log="array_regenie_step1_varprep.log"

export bed=array_regenie_step1_ldpruned_varprep.bed
export bim=array_regenie_step1_ldpruned_varprep.bim
export fam=array_regenie_step1_ldpruned_varprep.fam
export log2=array_regenie_step1_ldpruned_varprep.log

mv ${output_prune_in} ${output_prune_out} ${bed} ${bim} ${fam} ${log} ${log2} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
#Upload to GCP Bucket
!gsutil cp aux_scripts/3_REGENIE_PREP_step1_varprep.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-8" 
SCRIPT_NAME="3_REGENIE_PREP_step1_varprep.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"


dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "regenie_step1_varprep" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input sample_qc_pass_ids="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv" \
    --input variant_qc_pass_vars="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/2_VARIANT_QC/array/array_variantqc_pass_final_vars.txt" \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251107/array_regenie_step1_ldpruned_varprep.bim ../3_output/3_REGENIE_PREP/3_regenie_files/
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/REGENIE_PREP/20251107/array_regenie_step1_ldpruned_varprep.log ../3_output/3_REGENIE_PREP/3_regenie_files/

## HCM Case/Control Definition

This defines the HCM cases and the non-HCM controls for use in the GWAS itself as per the standard protocol documentation for HCM GWAS. 

In [ ]:
data = pd.read_csv('../3_output/3_REGENIE_PREP/1_flashPCA2/hcm_gwas_sample_qc_pass_PCs.tsv',sep='\t') #Import in the individuals passing sample QC

### HCM Cases

For cases, the definition is clinical diagnosis of HCM via ICD10 I42.1/42.2 or ICD9 425.1 codes
    - I42.2 diagnosis in the absence of other diagnosis codes requires 2 diagnoses for certainty.
    
The exclusion criteria is as per the .pptx.

In [ ]:
# dataset_69927227_condition_df.to_csv('../1_input/0_cohort_tsvs/hcm_pure.tsv',sep='\t',index=False)

In [ ]:
hcm_cases = pd.read_csv('../1_input/0_cohort_tsvs/hcm_pure.tsv',sep='\t') #This represents all HCM cases withOUT exclusion criteria applied
len(hcm_cases['person_id'].unique())

In [ ]:
#This separates the HCM cases who are diagnosed only by ICD10 42.2 and the others

# 1) Row-level match for the exact pair
pair_ok = (
    hcm_cases["source_vocabulary"].eq("ICD10CM")
    & hcm_cases["source_concept_code"].eq("I42.2")
)

# 2) For each person_id, check that *all* their rows match the pair i.e they have no other forms of diagnosis
only_pair_per_person = pair_ok.groupby(hcm_cases["person_id"], sort=False).transform("all")

# 3) Two outputs (row-level):
#    - df_icd422_only: all rows for people who ONLY ever have (ICD10CM, 42.2)
#    - df_other:       all rows for everyone else
hcm_cases_icd422_only = hcm_cases[only_pair_per_person].copy()
hcm_cases_other = hcm_cases[~only_pair_per_person].copy()

In [ ]:
#This extracts only the individuals in the hcm_cases_icd422_only who have at least 2 rows per individual
#This corresponds to at least 2 separate diagnoses per individual

mask_2plus = hcm_cases_icd422_only["person_id"].map(
    hcm_cases_icd422_only["person_id"].value_counts(sort=False)
).ge(2)

hcm_cases_icd422_only_2plus = hcm_cases_icd422_only[mask_2plus].copy()

In [ ]:
hcm_cases_icd422_only_1=hcm_cases_icd422_only[~mask_2plus].copy() #These individuals are excluded from CONTROLS
print(f"Number of HCM cases who only have I42.2 ICD10 diagnosis: {len(hcm_cases_icd422_only['person_id'].unique())}")
print(f"Number of HCM cases who only have I42.2 ICD10 diagnosis & have >= 2 separate diagnoses: {len(hcm_cases_icd422_only_2plus['person_id'].unique())}")
print(f"Number of HCM cases who only have I42.2 ICD10 diagnosis & have 1 diagnosis: {len(hcm_cases_icd422_only_1['person_id'].unique())}")

In [ ]:
#Create the output .tsv of HCM cases as per the case definition (i.e. after removing those individuals with only 1 I42.2)
hcm_cases_icd422filtered = pd.concat([hcm_cases_other, hcm_cases_icd422_only_2plus]).loc[:,'person_id'].drop_duplicates()
hcm_cases_icd422filtered.to_csv('../3_output/3_REGENIE_PREP/2_ids/hcm_cases_icd422filtered.tsv',sep='\t',index=False)
hcm_cases_icd422filtered.shape #This represents in the entire AoU V8 (not just the GWAS individuals)

In [ ]:
#This filters out the HCM cases by their intersection with the 303,927 individuals passing sample-level QC
hcm_cases_filtered = hcm_cases_icd422filtered[hcm_cases_icd422filtered.isin(data['person_id'])]
len(hcm_cases_filtered.unique())

In [ ]:
#This filters out the HCM cases by the exclusion criteria
## This hcm_case_exclusion represnets all individuals in AoU (with EHR/srWGS/array who pass the CASE exclusion criteria)
hcm_case_exclusion = pd.read_csv('../1_input/0_cohort_tsvs/hcm_gwas_case_exclusion_ids.tsv',sep='\t') 
hcm_cases_postexclusion = hcm_cases_filtered[~hcm_cases_filtered.isin(hcm_case_exclusion['person_id'])]
len(hcm_cases_postexclusion.unique())

In [ ]:
#Output the IDs
hcm_case_postexclusion_df = pd.DataFrame({'IID':hcm_cases_postexclusion})
hcm_case_postexclusion_df = hcm_case_exclusion_df.assign(FID=0)
hcm_case_postexclusion_df = hcm_case_exclusion_df.loc[:,['FID','IID']]
hcm_case_postexclusion_df.to_csv('../3_output/3_REGENIE_PREP/2_ids/hcm_cases_final_ids.tsv',sep='\t',index=False)

### Non-HCM Controls

For controls, the definition is individuals in the sampleQC pass who aren't HCM cases.
    
The exclusion criteria is as per the .pptx.

In [ ]:
hcm_controls_preexclusion = data.loc[~data['person_id'].isin(hcm_case_postexclusion_df['IID'])]
hcm_controls_preexclusion.shape

In [ ]:
#Exclude the individuals with one diagnosis of I42.2 (but not 2)
hcm_controls_postexclusion1 = hcm_controls_preexclusion.loc[~hcm_controls_preexclusion['person_id'].isin(hcm_cases_icd422_only_1['person_id'])]
hcm_controls_postexclusion1.shape

In [ ]:
# dataset_00411130_person_df.to_csv('../1_input/0_cohort_tsvs/hcm_gwas_control_exclusion_ids.tsv', sep='\t', index=False)

In [ ]:
#This inmports in the individuals in AoU who pass any of the CONTROL exclusion criteria 
hcm_controls_exclusion = pd.read_csv('../1_input/0_cohort_tsvs/hcm_gwas_control_exclusion_ids.tsv', sep='\t')

#This performs the filtering of those individuals against the CONTROL exclusion criteria
hcm_controls_postexclusion2 = hcm_controls_postexclusion1.loc[~hcm_controls_postexclusion1['person_id'].isin(hcm_controls_exclusion['person_id'])]
#Output the IDs
hcm_controls_postexclusion2_df = pd.DataFrame({'IID':hcm_controls_postexclusion2['person_id']})
hcm_controls_postexclusion2_df = hcm_controls_postexclusion2_df.assign(FID=0)
hcm_controls_postexclusion2_df = hcm_controls_postexclusion2_df.loc[:,['FID','IID']]
hcm_controls_postexclusion2_df.to_csv('../3_output/3_REGENIE_PREP/2_ids/hcm_controls_final_ids.tsv',sep='\t',index=False)
hcm_controls_postexclusion2_df.shape

## REGENIE Preparation

This prepares the covariate files + phenotype files for REGENIE.

## Covariate File

In [ ]:
#Import in the height/weight covariates

# ht_wt_df = dataset_59848456_measurement_df.loc[:,['person_id','standard_concept_name','measurement_datetime','value_as_number', 'unit_concept_name']]
# ht_wt_df = ht_wt_df.dropna(subset=['value_as_number'])
# ht_wt_df.to_csv('../1_input/0_cohort_tsvs/all_ht_wt_df.tsv',sep='\t', index=False)

ht_wt_df=pd.read_csv('../1_input/0_cohort_tsvs/all_ht_wt_df.tsv',sep='\t')
print(ht_wt_df['unit_concept_name'].value_counts())

In [ ]:
#Filter to single observation (earliest one) for each individual for height and weight

# Ensure datetime dtype
ht_wt_df = ht_wt_df.copy()
ht_wt_df["measurement_datetime"] = pd.to_datetime(ht_wt_df["measurement_datetime"])

# For each (person_id, standard_concept_name), keep the earliest measurement
idx = ht_wt_df.groupby(["person_id", "standard_concept_name"])["measurement_datetime"].idxmin()
first = ht_wt_df.loc[idx, ["person_id", "standard_concept_name", "value_as_number"]]

# Pivot to wide format and rename columns to height/weight
out = (
    first.pivot(
        index="person_id",
        columns="standard_concept_name",
        values="value_as_number"
    )
    .rename(columns={"Body height": "height", "Body weight": "weight"})
    .reset_index()
)

# Optional: ensure column order
out = out[["person_id", "height", "weight"]]
out.to_csv('../3_output/0_cohort_tsvs/all_ht_wt_processed.tsv', index=False, sep='\t')

In [ ]:
#Import in the batch covariate of srWGS source (i.e. saliva or blood)
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv ../1_input/0_cohort_tsvs/

In [ ]:
batch_cov = pd.read_csv('../1_input/0_cohort_tsvs/genomic_metrics.tsv', sep='\t').loc[:,['research_id', 'sample_source']]
batch_cov=batch_cov.rename(columns={'research_id':'person_id'})

In [ ]:
#Output the covariate file for REGENIE

##First import in the case IDs and control IDs and the main covariate df
hcm_case_postexclusion_df = pd.read_csv('../3_output/3_REGENIE_PREP/2_ids/hcm_cases_final_ids.tsv',sep='\t')
hcm_controls_postexclusion2_df = pd.read_csv('../3_output/3_REGENIE_PREP/2_ids/hcm_controls_final_ids.tsv',sep='\t')
data = pd.read_csv('../3_output/3_REGENIE_PREP/1_flashPCA2/hcm_gwas_sample_qc_pass_PCs.tsv',sep='\t')
data_filtered_cc = data.loc[data['person_id'].isin(pd.concat([hcm_case_postexclusion_df['IID'], hcm_controls_postexclusion2_df['IID']]))]
print(data_filtered_cc.shape)

In [ ]:
#Add the covariate of age at primary consent/enrollment
DATASET = %env WORKSPACE_CDR
df = pd.read_gbq(f'''
SELECT DISTINCT person_id, MIN(observation_date) AS primary_consent_date
FROM `{DATASET}.concept`
JOIN `{DATASET}.concept_ancestor` on concept_id = ancestor_concept_id
JOIN `{DATASET}.observation` on descendant_concept_id = observation_source_concept_id
WHERE concept_name = 'Consent PII' AND concept_class_id = 'Module'
GROUP BY 1''')
data_filtered_cc = data_filtered_cc.merge(df, how='left',on='person_id') #Add to the original df

In [ ]:
data_filtered_cc = data_filtered_cc.copy()
data_filtered_cc["primary_consent_date"] = pd.to_datetime(data_filtered_cc["primary_consent_date"], utc=True,errors="coerce")
data_filtered_cc["date_of_birth"] = pd.to_datetime(data_filtered_cc["date_of_birth"], utc=True, errors="coerce")

delta = data_filtered_cc["primary_consent_date"] - data_filtered_cc["date_of_birth"]
data_filtered_cc["age_at_primary_consent"] = (delta / pd.Timedelta(days=365.25)).where(delta.notna())

In [ ]:
##Add the covariates of height/weight (Physical Measurements)
data_filtered_cc_htwt = data_filtered_cc.merge(out, on='person_id', how='left')
#Add the batch covariate of srWGS source (either saliva or blood)
data_filtered_cc_htwt_batch = data_filtered_cc_htwt.merge(batch_cov, on='person_id', how='left')
data_filtered_cc_htwt_batch.isna().sum()

In [ ]:
#Remove individuals which are NA values for height/weight
data_filtered_cc_htwt_batch=data_filtered_cc_htwt_batch.dropna(subset=['height','weight'])
data_filtered_cc_htwt_batch.isna().sum()

In [ ]:
#Check each individual has one row
print(data_filtered_cc_htwt_batch.shape)
print(len(data_filtered_cc_htwt_batch['person_id'].unique()))

In [ ]:
regenie_cov = data_filtered_cc_htwt_batch.assign(FID=0)
col = regenie_cov.pop('FID') 
regenie_cov.insert(0, 'FID', col) 
regenie_cov=regenie_cov.rename(columns={'person_id':'IID'})
print(regenie_cov.shape)
regenie_cov=regenie_cov.drop(columns=['date_of_birth','primary_consent_date'])
regenie_cov['sample_source'] = np.where(regenie_cov['sample_source']=='WHOLE BLOOD', 'BLOOD','SALIVA')
regenie_cov=regenie_cov.drop(columns=['race'])

In [ ]:
regenie_cov.to_csv('../3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_cov.tsv',sep='\t',index=False)

## Phenotype File

In [ ]:
regenie_pheno = regenie_cov.copy()
regenie_pheno['hcm'] = np.where(regenie_pheno['IID'].isin(hcm_case_postexclusion_df['IID']),1,0)
regenie_pheno = regenie_pheno.loc[:,['FID','IID','hcm']]
print(sum(regenie_pheno['hcm']==1))
print(sum(regenie_pheno['hcm']==0))
regenie_pheno.to_csv('../3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_pheno.tsv',sep='\t',index=False)

In [ ]:
print(sum(regenie_pheno['IID'].isin(hcm_controls_postexclusion2_df['IID']))) #Number of controls

In [ ]:
regenie_pheno=pd.read_csv('../3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_regenie_pheno.tsv',sep='\t')
regenie_ids = regenie_pheno.loc[:,['FID','IID']]
regenie_ids.to_csv('../3_output/3_REGENIE_PREP/3_regenie_files/hcm_gwas_ids.tsv',sep='\t',index=False)

## Backup

In [ ]:
# Backup
!gsutil cp 3_REGENIE_PREP.ipynb {my_bucket}/HCM_GWAS_V2/2_scripts/
!gsutil -m cp ../3_output/3_REGENIE_PREP/3_regenie_files/* {my_bucket}/HCM_GWAS_V2/3_output/3_REGENIE_PREP/3_regenie_files/

In [ ]:
!gsutil -m cp -r ../3_output/3_REGENIE_PREP/ {my_bucket}/HCM_GWAS_V2/3_output/